# PushT rs-IMLE training on Colab GPU

Runs the exact same `imle_policy/train.py` entrypoint used locally, so results are directly
comparable (same code, same wandb project family) -- this notebook is just a driver, it does
not reimplement any training logic. See `colab/COLAB_WORKFLOW.md` in this repo for the full
written walkthrough (what to upload, how to bring a trained model back to the local machine,
why quantization usually isn't needed for local eval).

**Runtime**: Runtime -> Change runtime type -> GPU (T4 is fine; A100 if you have Pro and want
it faster). Colab GPUs have far more VRAM (T4: 15GB, A100: 40GB) than the local MX450 (2GB), so
this notebook defaults to the **original, full-size architecture**
(`down_dims=[256,512,1024]`, `batch_size=128`) that doesn't fit locally -- that's the point of
running here in parallel: a third comparison point (original capacity) alongside the two
capacity-constrained local runs, using the same code.

## 1. Mount Google Drive
Checkpoints get saved here, not to the Colab VM disk, because Colab VMs are ephemeral --
everything on them is lost when the runtime disconnects/recycles (which *will* happen on a
run this long). Drive survives across sessions, so a disconnected run can be resumed in a
fresh runtime with `--resume_from` pointed at the same Drive path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/imle_policy_colab'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/saved_weights', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 2. Clone your fork
`saved_weights/`, `datasets/`, and `wandb/` are all gitignored in this repo (checked locally --
they're not in the git history), so cloning gets code only. That's deliberate: code goes
through git (so local and Colab always run identical logic), data and weights go through Drive
(too large / not meant to be versioned).

In [ ]:
GITHUB_USER = 'santhoshetty'  # your fork
!git clone https://github.com/{GITHUB_USER}/imle_policy.git /content/imle_policy
%cd /content/imle_policy
!git log --oneline -5

## 3. Install dependencies
`pyproject.toml`'s full dependency set pulls in robosuite/d4rl/mujoco/dm_control (for other
tasks this repo supports) which are heavy and finicky to build on a fresh Colab VM. PushT only
needs a subset -- installing that directly is faster and more reliable than `pip install -e .`
here. If you later want to run a different task's eval on Colab too, fall back to the full
`pip install -e .` and expect it to take a while.

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers wandb pymunk 'gym==0.23.1' 'gymnasium==0.29.1' pygame shapely \
    scikit-image opencv-python-headless 'imageio[ffmpeg]' numpy

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 4. Get the dataset onto Drive (one-time)
`imle_policy/datasets/pusht.pkl` is ~710MB, too big to be worth re-uploading every session.
Do this **once**; every future Colab session just points at the Drive copy.

Upload it to `MyDrive/imle_policy_colab/datasets/pusht.pkl` any way you like -- the Drive web
UI, `rclone`, or Colab's file upload widget (slow over a browser for 710MB, but works). Once
it's there, this cell just symlinks it into the expected local path.

In [ ]:
DRIVE_DATASET = f'{DRIVE_ROOT}/datasets/pusht.pkl'
assert os.path.exists(DRIVE_DATASET), (
    f'Upload pusht.pkl to {DRIVE_DATASET} first (Drive web UI is easiest), then re-run this cell.'
)

os.makedirs('/content/imle_policy/imle_policy/datasets', exist_ok=True)
local_link = '/content/imle_policy/imle_policy/datasets/pusht.pkl'
if not os.path.exists(local_link):
    os.symlink(DRIVE_DATASET, local_link)
print('dataset ready at', local_link)

## 5. wandb login
Same account/project family as the local runs, so Colab and local experiments show up
side-by-side for comparison without any extra plumbing.

In [ ]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted

## 6. Point saved_weights at Drive, then launch training
`train.py` writes checkpoints to `saved_weights/<run_name>/` relative to its working directory
-- symlinking that directory to Drive means every checkpoint (including the frequent
lightweight ones) lands durably on Drive automatically, no extra copy step, and survives a
disconnected runtime.

**This is the systematic part**: change only the flags in the `!python train.py ...` line to
run a different experiment. Everything else in this notebook stays the same. A few ready-made
variants are given below the main cell -- uncomment the one you want instead of the default.

In [ ]:
os.makedirs(f'{DRIVE_ROOT}/saved_weights', exist_ok=True)
local_weights = '/content/imle_policy/imle_policy/saved_weights'
if os.path.islink(local_weights) or os.path.exists(local_weights):
    if os.path.islink(local_weights):
        os.remove(local_weights)
os.symlink(f'{DRIVE_ROOT}/saved_weights', local_weights)
print('saved_weights ->', os.path.realpath(local_weights))

In [ ]:
%cd /content/imle_policy/imle_policy

# Default: original, full-size architecture -- the comparison point that doesn't fit locally.
# n_samples_per_condition left at the repo default (20) to match both local runs.
!python train.py --task pusht --method rs_imle \
    --wandb_run_name pusht_colab_original_capacity \
    --down_dims 256,512,1024 \
    --batch_size 128

# --- Other experiments: comment out the cell above, uncomment one of these instead ---

# Re-test batch-global rejection at ORIGINAL capacity (it failed at 7.5M/22.6M params on
# PushT's narrow action manifold for a documented reason -- see IMPROVEMENTS.md -- unlikely to
# flip with more capacity alone, since the failure mode was about early-training cross-context
# corruption, not capacity, but cheap to confirm on a GPU that isn't otherwise busy):
# !python train.py --task pusht --method rs_imle \
#     --wandb_run_name pusht_colab_batch_global_original_capacity \
#     --down_dims 256,512,1024 --batch_size 128 --use_batch_global_rejection

# Resume an interrupted Colab run (Drive path persists across a disconnected runtime):
# !python train.py --resume_from saved_weights/<run_name>/latest_checkpoint.pth

## 7. If the runtime disconnects
Re-run cells 1-3 and 6's symlink cell in a fresh runtime (dataset symlink in cell 4 too), then
resume:
```python
!python train.py --resume_from saved_weights/pusht_colab_original_capacity_.../latest_checkpoint.pth
```
Everything (model, optimizer, EMA, RNG state, wandb run id) is restored -- same mechanism as
the local resume workflow, because it's the same code.

## 8. Bring a checkpoint back to the local machine
Nothing to zip or export specially -- Drive already has it (that's what the symlink in step 6
was for). On the **local** machine: install/use the Google Drive desktop app, or
`rclone copy 'gdrive:imle_policy_colab/saved_weights/<run_name>' ./saved_weights/<run_name>`,
or just download the specific files you need (e.g. `latest_checkpoint.pth`,
`ema_net_weights_epoch<N>.pth`) from the Drive web UI. See `colab/COLAB_WORKFLOW.md` for the
full pull-and-eval steps, including why quantization usually isn't needed just to run eval
locally even though training the full-size model doesn't fit on the local GPU.